# RadCluster_2_1 — Digital-Twin Campaign Control

Run this notebook **on each machine**. It configures the run, verifies the
toolchain, launches the worker, and gives you a live view of where the
campaign stands against the experimental data.

| cell | what it does |
|---|---|
| **1 Configure** | every knob for this machine in one place |
| **2 Build check** | verifies the C++ solver; compiles it if missing *or stale* |
| **3 Agreement check** | confirms this machine reproduces the reference numbers |
| **4 Launch** | starts the worker in the background |
| **5 Monitor** | live progress, timing, remaining compute, results vs experiment |
| **6 Graceful stop** | halt without losing work |
| **7 Restart** | resume and keep everything already computed |
| **8 Tally** | merge all machines → Sobol indices |

**The one rule:** every machine must run the *same commit* and the *same
design file*. Cell 3 enforces it; cell 5 warns if it ever drifts.

## 1 — Configure

In [ ]:
from pathlib import Path
import os, sys, json, subprocess

HERE = Path.cwd() if Path.cwd().name == 'digital_twin' else Path('RadCluster_2_1/digital_twin')
sys.path.insert(0, str(HERE))
import campaign_ops as ops

# ── THIS MACHINE ─────────────────────────────────────────────────────────
MACHINE     = 0        # 0,1,2,3 — MUST be unique across the four machines
N_MACHINES  = 4
WORKERS     = max(1, (os.cpu_count() or 4) - 2)   # single-threaded workers

# ── THE RUN ──────────────────────────────────────────────────────────────
DESIGN      = HERE / 'design' / 'T2_design_v1.csv'
RESULTS     = HERE / 'results'
I_GRID      = 800      # set from the T0.2 grid study — see note below
V_GRID      = 600
DOSE        = 0.1      # dpa, the Tier-2 LF dose
EQUATIONS   = 'discrete'   # 'bin_moment' once it is validated (much faster)
RTOL        = 1e-6
TIMEOUT_S   = 3600     # per row
LIMIT       = 0        # >0 = smoke test on the first N rows only

env = ops.environment()
print(f"machine   {env['machine_id']}  ({env['cpu_count']} cores, using {WORKERS} workers)")
print(f"branch    {env['branch']}  @ {env['git_sha'][:12]}" + ('  *** WORKTREE DIRTY ***' if env['worktree_dirty'] else ''))
print(f"design    {DESIGN.name}")
print(f"run       I={I_GRID} V={V_GRID} {DOSE} dpa {EQUATIONS} rtol={RTOL}")
if env['worktree_dirty']:
    print('\n  Uncommitted changes: this machine may not match the others.')
    print('  Commit or stash before a production run.')

## 2 — Build check (auto-builds if needed)

The solver binary is **not** in git — each machine compiles its own. This also
catches a *stale* binary: if any `.cpp`/`.h`/`CMakeLists.txt` is newer than
`solver.exe`, it rebuilds. That is the case that silently produces results from
code you thought you had replaced.

In [ ]:
info = ops.ensure_solver()          # ops.ensure_solver(force=True) to rebuild anyway
info

## 3 — Machine agreement check

Runs a fixed ~30 s probe and compares 12 quantities against the committed
reference at `rtol=1e-9`. CVODE is deterministic for a fixed binary, so a
nonzero difference means a genuinely different build — not noise.

**Do not start a machine that fails this.** Its rows cannot be pooled with the
others, and the failure will not be visible in the physics.

In [ ]:
r = subprocess.run([sys.executable, str(HERE / 'check_machine.py')],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print('AGREEMENT OK' if r.returncode == 0 else f'*** FAILED (rc={r.returncode}) — do not launch ***')

## 4 — Launch the worker

Starts in the background so the monitor cell stays usable. Safe to re-run:
rows already completed are skipped.

In [ ]:
ops.clear_stop()          # make sure no stale STOP flag is present
RESULTS.mkdir(parents=True, exist_ok=True)
log = RESULTS / f'worker_machine{MACHINE}.log'
cmd = [sys.executable, '-u', str(HERE / 'run_ensemble.py'),
       '--design', str(DESIGN), '--machine', str(MACHINE), '--of', str(N_MACHINES),
       '--workers', str(WORKERS), '--I', str(I_GRID), '--V', str(V_GRID),
       '--dose', str(DOSE), '--equations', EQUATIONS, '--rtol', str(RTOL),
       '--timeout-s', str(TIMEOUT_S)] + (['--limit', str(LIMIT)] if LIMIT else [])
print(' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=open(log, 'a'), stderr=subprocess.STDOUT)
print(f'\nlaunched pid {proc.pid}, logging to {log}')

## 5 — Live monitor

Refreshes in place. **Interrupt the kernel to stop watching — the campaign is
unaffected.** Shows coverage, per-machine progress, per-row timing, remaining
core-hours and ETA, why rows were rejected, provenance drift, and where the
ensemble sits against each experimental band.

On the observables table: a low *in band* fraction early on is normal — the
prior box is deliberately wide. Stuck at 0% after several hundred rows means
the box does not contain the data, which is itself a result.

In [ ]:
ops.watch(DESIGN, RESULTS, n_machines=N_MACHINES,
          workers_per_machine=WORKERS, interval=60)

In [ ]:
# One-shot snapshot instead of the live loop
st = ops.campaign_status(DESIGN, RESULTS, N_MACHINES, WORKERS)
ops.render_status(st, ops.load_targets())

### 5b — Inspect suspicious rows

If the monitor shows something unphysical, look before you stop. This lists the
worst offenders by category so you can tell a bad *parameter region* (a
result — record it) from a bad *model* (a reason to stop).

In [ ]:
import numpy as np
recs = list(ops.load_results(RESULTS).values())
bad  = [r for r in recs if not r.get('solver_rc') and not r.get('admissible')]
print(f'{len(bad)} inadmissible of {len(recs)}\n')
for r in sorted(bad, key=lambda z: -(z.get('pile_100') or 0))[:10]:
    print(f"  row {r['row_id']:6d} {r['condition']} pile100={r.get('pile_100')} "
          f"occ100={r.get('occ_100'):.3f} d100={r.get('d_100_nm'):.2f} "
          f"dose={r.get('dose_reached'):.3f} dFP={r.get('delta_FP'):.1e}")
print('\nfailed rows:')
for r in [x for x in recs if x.get('solver_rc')][:10]:
    print(f"  row {r['row_id']:6d}  {r.get('error','')[:110]}")

## 6 — Graceful stop

Writes a sentinel the worker checks between rows. It **stops submitting new
work, lets in-flight rows finish and be written, then exits cleanly**. Nothing
is lost and nothing partial is written.

Do *not* kill the kernel or the process — that discards every row currently
being computed (up to `WORKERS` of them, each possibly an hour of compute).

In [ ]:
ops.request_stop('unphysical d_100 seen in the monitor')   # put the real reason here
# then watch the tail of the log until it prints STOPPED:
print(open(log).read()[-1200:])

## 7 — Change something, then restart

Everything already computed is kept — `run_ensemble` skips `row_id`s already in
this machine's `.jsonl`.

**But resuming is only a benefit if the new rows are comparable to the old
ones.** If you change code, the solver, the workbook or the design, the runner
*refuses* to append (exit 3) and tells you which hash moved. Two honest options:

* **The change affects results** (a kernel, a parameter, the grid) — archive the
  old rows and start that machine's file clean. The elapsed compute is not
  wasted: it is preserved as a labelled prior campaign.
* **The change cannot affect results** (a comment, the README) — re-run with
  `--allow-mixed`.

Changing `I_GRID`, `DOSE` or `EQUATIONS` **always** invalidates prior rows —
they are not part of `θ`, so nothing else records that they moved.

In [ ]:
# Archive this machine's rows under a label, keeping them for the record
import shutil, time as _t
src = RESULTS / f'{DESIGN.stem}_machine{MACHINE}.jsonl'
if src.exists():
    tag = input('label for the archived campaign (e.g. pre-Ea0-fix): ').strip() or _t.strftime('%Y%m%d_%H%M')
    dst = RESULTS / 'archive' / f'{src.stem}__{tag}.jsonl'
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dst))
    print(f'archived -> {dst}')
else:
    print('nothing to archive')

In [ ]:
ops.clear_stop()
# re-run cell 2 (rebuild if you touched C++), then cell 3, then cell 4.

## 8 — Tally across machines

Each machine writes its own file, so they never conflict; `git pull` (or a
shared drive) brings them together. Merging is keyed on `row_id`, so it is
order-independent and idempotent — safe to run on a partial campaign and again
later.

In [ ]:
r = subprocess.run([sys.executable, str(HERE / 'merge_and_sobol.py'),
                    '--design', str(DESIGN), '--results', str(RESULTS),
                    '--out', str(HERE / 'report')],
                   capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-2000:])